In [ ]:
pip install matminer

In [ ]:
import pandas as pd
import numpy as np
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.structure import DensityFeatures
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor

In [ ]:
df = load_dataset('elastic_tensor_2015')

In [ ]:
df = df.drop(['material_id','compliance_tensor','elastic_tensor','elastic_tensor_original'], axis=1)
df = df.reset_index(drop=True)

In [ ]:
#Formula features via ElementProperty (magpie)
# df_formula_feats

comp_featurizer = ElementProperty.from_preset('magpie')

def featurize_formula(formula):
    return comp_featurizer.featurize(formula)

from pymatgen.core import Composition
df['composition'] = df['formula'].apply(Composition)

df_formula_feats = pd.DataFrame(df['composition'].apply(featurize_formula).to_list(),
                                columns=comp_featurizer.feature_labels())



/usr/local/lib/python3.11/dist-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


In [ ]:
df_formula_feats.columns

Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'MagpieData range GSmagmom', 'MagpieData mean GSmagmom',
       'MagpieData avg_dev GSmagmom', 'MagpieData mode GSmagmom',
       'MagpieData minimum SpaceGroupNumber',
       'MagpieData maximum SpaceGroupNumber',
       'MagpieData range SpaceGroupNumber', 'MagpieData mean SpaceGroupNumber',
       'MagpieData avg_dev SpaceGroupNumber',
       'MagpieData mode SpaceGroupNumber'],
      dtype='object', length=132)

In [ ]:
# Initialize structure featurizers
from pymatgen.core import Structure
from matminer.featurizers.structure import GlobalSymmetryFeatures

#from cif string to structure
df['structure'] = df['cif'].apply(lambda cif_str: Structure.from_str(cif_str, fmt='cif'))

feature_labels = DensityFeatures().feature_labels()
df_structure_feats = pd.concat([DensityFeatures().featurize_dataframe(df, 'structure')[feature_labels],df[['structure','composition']]],axis =1)


df_structure_feats = GlobalSymmetryFeatures().featurize_dataframe(df_structure_feats, 'structure')

add = pd.DataFrame(df_structure_feats['composition'].apply(featurize_formula).to_list(),
                                columns=comp_featurizer.feature_labels())



/usr/local/lib/python3.11/dist-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.11/dist-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.11/dist-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 16 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.11/dist-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issu

DensityFeatures:   0%|          | 0/1181 [00:00<?, ?it/s]

GlobalSymmetryFeatures:   0%|          | 0/1181 [00:00<?, ?it/s]

In [ ]:
df_structure_feats = pd.concat([df_structure_feats, add], axis=1)
df_structure_feats = df_structure_feats.drop(['structure','crystal_system','composition','is_centrosymmetric'],axis=1)
# df.dtypes[df.dtypes == 'object']
# df[df.duplicated()]



In [ ]:
y = df['K_VRH']

In [ ]:
for col in df_structure_feats.columns:
    print(col)


density
vpa
packing fraction
spacegroup_num
crystal_system_int
n_symmetry_ops
MagpieData minimum Number
MagpieData maximum Number
MagpieData range Number
MagpieData mean Number
MagpieData avg_dev Number
MagpieData mode Number
MagpieData minimum MendeleevNumber
MagpieData maximum MendeleevNumber
MagpieData range MendeleevNumber
MagpieData mean MendeleevNumber
MagpieData avg_dev MendeleevNumber
MagpieData mode MendeleevNumber
MagpieData minimum AtomicWeight
MagpieData maximum AtomicWeight
MagpieData range AtomicWeight
MagpieData mean AtomicWeight
MagpieData avg_dev AtomicWeight
MagpieData mode AtomicWeight
MagpieData minimum MeltingT
MagpieData maximum MeltingT
MagpieData range MeltingT
MagpieData mean MeltingT
MagpieData avg_dev MeltingT
MagpieData mode MeltingT
MagpieData minimum Column
MagpieData maximum Column
MagpieData range Column
MagpieData mean Column
MagpieData avg_dev Column
MagpieData mode Column
MagpieData minimum Row
MagpieData maximum Row
MagpieData range Row
MagpieData me

In [ ]:
df_structure_feats.columns

Index(['density', 'vpa', 'packing fraction', 'spacegroup_num',
       'crystal_system_int', 'n_symmetry_ops', 'MagpieData minimum Number',
       'MagpieData maximum Number', 'MagpieData range Number',
       'MagpieData mean Number',
       ...
       'MagpieData range GSmagmom', 'MagpieData mean GSmagmom',
       'MagpieData avg_dev GSmagmom', 'MagpieData mode GSmagmom',
       'MagpieData minimum SpaceGroupNumber',
       'MagpieData maximum SpaceGroupNumber',
       'MagpieData range SpaceGroupNumber', 'MagpieData mean SpaceGroupNumber',
       'MagpieData avg_dev SpaceGroupNumber',
       'MagpieData mode SpaceGroupNumber'],
      dtype='object', length=138)

In [ ]:

# Train-test split function
def split_scale(X):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

# Models to try
models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(),
    "AdaBoost": AdaBoostRegressor(random_state=42)
}


In [ ]:

# Function to train, evaluate and return best model and score
def train_evaluate(X):
    best_r2 = -np.inf
    best_model_name = None
    best_model = None
    for name, model in models.items():
        X_train, X_test, y_train, y_test, scaler = split_scale(X)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        print(f"{name} R²: {r2:.4f}")
        if r2 > best_r2:
            best_r2 = r2
            best_model_name = name
            best_model = model
            best_scaler = scaler
    return best_model_name, best_model, best_r2, best_scaler


In [ ]:
#f_name, f_model, f_r2, f_scaler = train_evaluate(df_formula_feats)
s_name, s_model, s_r2, s_scaler = train_evaluate(df_structure_feats)


 # Summary:
best_overall ={'Structure': (s_r2, s_name, s_model, s_scaler)}#'Formula': (f_r2, f_name, f_model, f_scaler),

for key, value in best_overall.items():
    print(f"{key}: {value[0]:.4f} ({value[1]})")


Decision Tree R²: 0.8560
Random Forest R²: 0.9163
Gradient Boosting R²: 0.9316
SVR R²: 0.3465
AdaBoost R²: 0.8671
Structure: 0.9316 (Gradient Boosting)


In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200],           # Reduced from 3 to 2
    'learning_rate': [0.05, 0.1],         # Reduced from 3 to 2
    'max_depth': [3, 5],                  # Reduced from 3 to 2
    'subsample': [0.8]
}
gb = GradientBoostingRegressor(random_state=42)
grid_search = GridSearchCV(gb, param_grid, cv=5, scoring='r2', n_jobs=-1)
X_train, X_test, y_train, y_test, scaler = split_scale(df_structure_feats)
grid_search.fit(X_train, y_train)
print(f"Best params: {grid_search.best_params_}, Best R²: {grid_search.best_score_:.4f}")

Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}, Best R²: 0.9393


In [ ]:

# Save best model and scaler
from google.colab import drive
drive.mount('/content/drive')
import joblib
joblib.dump(grid_search, "/content/drive/MyDrive/best_structure_model4.pkl")
joblib.dump(scaler, "/content/drive/MyDrive/best_structure_scaler4.pkl")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['/content/drive/MyDrive/best_structure_scaler4.pkl']

In [ ]:
from google.colab import drive
import joblib
drive.mount('/content/drive')
joblib.dump(f_model, "/content/drive/MyDrive/best_formula_model2.pkl")
joblib.dump(f_scaler, "/content/drive/MyDrive/best_formula_scaler2.pkl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['/content/drive/MyDrive/best_formula_scaler2.pkl']

In [ ]:
pip show matminer pymatgen


Name: matminer
Version: 0.9.3
Summary: matminer is a library that contains tools for data mining in Materials Science
Home-page: https://github.com/hackingmaterials/matminer
Author: Anubhav Jain
Author-email: anubhavster@gmail.com
License: modified BSD
Location: /usr/local/lib/python3.11/dist-packages
Requires: monty, numpy, pandas, pymatgen, pymongo, requests, scikit-learn, sympy, tqdm
Required-by: 
---
Name: pymatgen
Version: 2025.6.14
Summary: Python Materials Genomics is a robust materials analysis code that defines core object representations for structures
Home-page: https://pymatgen.org
Author: 
Author-email: Pymatgen Development Team <ongsp@ucsd.edu>
License: MIT
Location: /usr/local/lib/python3.11/dist-packages
Requires: bibtexparser, joblib, matplotlib, monty, networkx, numpy, orjson, palettable, pandas, plotly, requests, ruamel.yaml, scipy, spglib, sympy, tabulate, tqdm, uncertainties
Required-by: matminer


In [ ]:
from google.colab import drive
import json

# Mount Google Drive
drive.mount('/content/drive')

# Suppose df_structure_feats is your dataframe with feature columns during training
feature_cols = df_structure_feats.columns.tolist()

# Save to your Google Drive path
feature_cols_path = "/content/drive/MyDrive/structure_feature_cols2.json"
with open(feature_cols_path, "w") as f:
    json.dump(feature_cols, f)

print(f"Feature columns saved to {feature_cols_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Feature columns saved to /content/drive/MyDrive/structure_feature_cols2.json
